# Walmart Retail Time Series Forecasting

In this exercise we use an **LSTM** to forecast item-level sales from the Walmart M5 competition dataset.

**Dataset:** `walmart_retail/` directory containing:
- `sales_train_validation.csv` -- daily unit sales per item per store (1,913 days)
- `calendar.csv` -- date-to-day mapping with event info
- `sell_prices.csv` -- weekly item prices (not used here)
- `sales_train_evaluation.csv` -- extended sales data (not used here)

**Prepared subset.** The sales file is 120 MB. Its first 3,049 rows are all store CA_1, so this notebook reads only the first three rows -- exactly the three items analysed below -- and loads in about a second. Set `SUBSET_ROWS = None` in Section 2 to explore the full file.

**Learning objectives:**
1. Explore a real-world retail time series dataset.
2. Select and visualize individual item/store sales trajectories.
3. Prepare sequential data for LSTM forecasting with a training / validation / test split in time.
4. Build, train (with early stopping on validation) and evaluate an LSTM sales forecaster.
5. Compare it with the weekly seasonal naive and a 28-day mean on the same test days.

**Evaluation design.** Windows are split by target day: the first 70% for training, the next 10% for validation (early stopping and every comparison during development) and the last 20% for the final test, used once. Many days have zero sales, so we report MAE (and RMSE), not MAPE. Runtime: under half a minute on an Apple M3 Max laptop CPU; allow up to twice as long on older laptops.

---
## 1. Imports

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights, dropout and DataLoader shuffling

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
# YOUR CODE HERE
# Hint: Load the prepared subset of sales_train_validation.csv from the walmart_retail/ directory:
#   SUBSET_ROWS = 3   # the first three rows are the three CA_1 items used below
#   sales = pd.read_csv("walmart_retail/sales_train_validation.csv", nrows=SUBSET_ROWS)
# Print shape, first/last column names, and head(3)

In [ ]:
# YOUR CODE HERE
# Hint: Load calendar.csv and display its shape and head()

In [ ]:
# YOUR CODE HERE
# Hint:
# day_cols = [c for c in sales.columns if c.startswith("d_")]
# Print number of day columns, unique items, unique stores
# (with the subset these describe the rows loaded; the full file has 3,049 items in 10 stores)

---
## 3. Select Items and Stores for Analysis

The full dataset is very large. We will select a few specific items from specific stores to keep things manageable: the first three items of store CA_1.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Filter for store "CA_1": store_data = sales[sales["store_id"] == "CA_1"].head(3)
# 2. Extract daily sales: item_sales = store_data[day_cols].values.astype(np.float32)
# 3. Save item names: item_names = store_data["id"].values

---
## 4. Visualize Sales Over Time

In [ ]:
# YOUR CODE HERE
# Hint: Create a 3-row subplot figure, one row per item
# Plot item_sales[i] for each item with title = item name

In [ ]:
# YOUR CODE HERE
# Hint: Plot 7-day rolling average for each item on the same plot
# Use pd.Series(item_sales[i]).rolling(window=7).mean()

---
## 5. Prepare Data: Focus on One Item

We will train an LSTM to forecast sales for the first selected item, one day ahead from the past 28 days.

Each window belongs to the period that contains its **target** day: the first 70% of windows for training, the next 10% for validation and the last 20% for the test. Validation and test windows may read the preceding 28 observed days as inputs -- that is what a deployed forecast sees -- but no training target lies after the training period. The scaler is fitted on the days used by the training windows only.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. series = item_sales[0].reshape(-1, 1)
# 2. Normalize with MinMaxScaler(feature_range=(0, 1)), fitted on the TRAINING days only:
#       SEQ_LENGTH = 28
#       train_sequence_count = int((len(series) - SEQ_LENGTH) * 0.7)
#       training_raw_end = SEQ_LENGTH + train_sequence_count
#       scaler.fit(series[:training_raw_end])
#       series_scaled = scaler.transform(series)   # all days, same fitted scaler

In [ ]:
def create_sequences(data, seq_length):
    """Sliding window to create (input, target) pairs."""
    # YOUR CODE HERE
    # Hint:
    # Loop i from 0 to len(data)-seq_length:
    #   X.append(data[i : i + seq_length])
    #   y.append(data[i + seq_length])
    pass


SEQ_LENGTH = 28  # use past 28 days (4 weeks) to predict next day

X, y = create_sequences(series_scaled, SEQ_LENGTH)
print(f"X shape: {X.shape}, y shape: {y.shape}")

# Chronological split by target day: 70% train, 10% validation, 20% test
train_end = int(len(X) * 0.7)
val_end = int(len(X) * 0.8)
assert train_end + SEQ_LENGTH == training_raw_end  # the scaler saw training days only
X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"Train: {X_train.shape[0]}, Validation: {X_val.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# YOUR CODE HERE
# Hint: Convert X_train, y_train, X_val, y_val to PyTorch tensors (float32):
# X_train_t, y_train_t, X_val_t, y_val_t
# Create the training DataLoader: BATCH_SIZE = 32, shuffle=True

### Baselines on the same days

Simple forecasts read from the same input windows, so they have exactly the same targets as the LSTM:
- **Seasonal naive (7 days)**: the sales on the same weekday one week before the target, `X[:, -7]`.
- **28-day mean**: the mean of the 28 input days.
- **Zero (no sales)**: worth including for intermittent sales. MAE rewards the median, and this item sells nothing on most days; the LSTM, trained with MSE, aims at the mean instead.

We score the validation days now; the test days are scored once, in Section 8.

In [ ]:
# YOUR CODE HERE
# Hint (later cells reuse these names):
# 1. to_units(z): scaler.inverse_transform(np.asarray(z).reshape(-1, 1)).ravel()
# 2. score(y_true, y_pred): return {"MAE": ..., "RMSE": ...}
# 3. baseline_forecasts(X): return {"Seasonal naive (7 days)": to_units(X[:, -7, 0]),
#                                   "28-day mean": to_units(X[:, :, 0].mean(axis=1)),
#                                   "Zero (no sales)": np.zeros(len(X))}
# 4. y_val_units = to_units(y_val)
#    val_scores = {name: score(y_val_units, pred) for name, pred in baseline_forecasts(X_val).items()}
#    pd.DataFrame(val_scores).T.round(3)

---
## 6. Build LSTM for Sales Forecasting

In [ ]:
class SalesLSTM(nn.Module):
    """LSTM for univariate sales forecasting."""

    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        # YOUR CODE HERE
        # Hint:
        # self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
        #                     batch_first=True, dropout=dropout)
        # self.fc = nn.Linear(hidden_size, 1)
        pass

    def forward(self, x):
        # YOUR CODE HERE
        # Hint: LSTM forward -> take last time step -> FC layer
        pass


model = SalesLSTM().to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 7. Train and Evaluate

Train on the training windows and stop early when the validation loss has not improved for `PATIENCE` epochs; the weights of the best epoch (a deep copy) are restored before any evaluation. Every comparison in this section uses the validation days.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. criterion = nn.MSELoss(), optimizer = optim.Adam(..., lr=1e-3)
# 2. Train for at most NUM_EPOCHS=100 epochs with early stopping, PATIENCE=10
# 3. Standard training loop: zero_grad, forward, loss, backward, step; track train_losses
# 4. After each epoch compute the validation loss on X_val_t (model.eval(), torch.no_grad())
# 5. Keep best_state = copy.deepcopy(model.state_dict()) and best_epoch when it improves
# 6. After the loop, restore the best weights: model.load_state_dict(best_state)

In [ ]:
# YOUR CODE HERE
# Hint: Plot train_losses and val_losses; mark the chosen epoch with plt.axvline(best_epoch)

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. predict_units(model, X): model.eval(), torch.no_grad(), predict, return to_units(...)
# 2. val_scores["LSTM"] = score(y_val_units, predict_units(model, X_val))
# 3. pd.DataFrame(val_scores).T.round(3): does the LSTM beat the baselines on validation?

---
## 8. Final Test Evaluation and Predictions

This is the final evaluation, run once after every choice was made on validation: the baselines and the LSTM are scored on the same test days.

In [ ]:
# YOUR CODE HERE
# Hint (run once, at the end):
# 1. y_actual = to_units(y_test)
# 2. test_forecasts = baseline_forecasts(X_test); test_forecasts["LSTM"] = predict_units(model, X_test)
#    y_pred = test_forecasts["LSTM"]
# 3. Score every method on the same test days and show the table (baselines first)

In [ ]:
# YOUR CODE HERE
# Hint: Plot actual vs predicted sales on the test set

---
## 9. Summary

In this notebook we:

1. Loaded a prepared subset of the Walmart M5 retail dataset and explored its structure.
2. Selected specific items from a single store for analysis.
3. Visualized raw daily sales and 7-day rolling averages.
4. Prepared sliding-window sequences (28 days to predict 1 day ahead), split by target day into training, validation and test.
5. Built a 2-layer LSTM and trained it with early stopping on the validation loss.
6. Compared it with the weekly seasonal naive, a 28-day mean and a zero forecast -- on validation during development, and once on the test days.

**Key takeaways:**
- Retail sales data can be very noisy with many zero-sales days; MAPE is undefined there, so use MAE.
- Simple baselines are hard to beat on intermittent single-item sales. Say so plainly when it happens.
- MAE rewards the median: for an item that sells nothing on most days, a zero forecast can win on MAE. Choose the metric to match the decision the forecast supports.
- Caveat: this item sells nothing in its first 901 days, and `sell_prices.csv` lists no price for it before week 11325 (day 897): it was not yet on sale. A large part of the training period is therefore structural zeros, not demand.
- The sliding-window length should reflect the periodicity of the data (e.g., weekly patterns -> at least 7 days).
- More sophisticated approaches could incorporate calendar features (holidays, weekdays), price data, and hierarchical structure.
- For production forecasting, consider ensembling with traditional methods (e.g., ARIMA, exponential smoothing).